# Pseudo labeled data cleanup

## 1. Notebook setup
### 1.1. Imports

In [1]:
import pandas as pd
import configuration as config

### 1.2. Configuration

In [2]:
raw_data_path     = config.MERGED_CSV_PATH
cleaned_data_path = config.CLEANED_CSV_PATH

label_features = [
    'ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA',
    'PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture'
]

## 2. Data inspection
### 2.1. Load pseudolabeled data

In [ ]:
data = pd.read_csv(raw_data_path)

# Keep only pseudolabeled data (not the original gold set)
#data = data[data['label_source'] != 'gold']

# Discard the two studies that were unextractable/produced errors
data = data[data['StudyInstanceUID'] != '1.2.826.0.1.3680043.8.498.12166235678892893545054546864889743251']
data = data[data['StudyInstanceUID'] != '1.2.826.0.1.3680043.8.498.91168270990167204047014161851340054645']

# Get study ID for 'gold' labels
gold_study_ids = data.loc[data['label_source'] == 'gold', 'StudyInstanceUID'].unique()
len(gold_study_ids)

58

In [5]:
gold_studies = data[data['StudyInstanceUID'].isin(gold_study_ids)]
len(gold_studies)

58

### 2.2. Inspect

In [ ]:
data.info()

In [ ]:
data.head().T

In [ ]:
data.tail().T

### 2.3. Label frequency

In [ ]:
for label in label_features:
    display(data[label].value_counts())

In [ ]:
mask = data[label_features].isin([1]).any(axis=1)
normal_scans = data[~mask]
print(f'Total normal scans: {len(normal_scans)}')

## 3. Set data types

In [ ]:
data[label_features] = data[label_features].astype(int)
data.info()

## 4. Add reports

In [ ]:
reports_df = pd.read_csv(config.TRAIN_CSV)
reports_df.info()

In [ ]:
data['Report'] = reports_df['StudyInstanceUID'].map(reports_df.set_index('StudyInstanceUID')['Report'])

In [ ]:
data.info()

In [ ]:
data.head(3).T

## 5. Drop `pseudo_status` and `label_source`

In [ ]:
data.drop('pseudo_status', axis=1, inplace=True)
# data.drop('label_source', axis=1, inplace=True)

## 6. Check repeated report strings

In [ ]:
repeated_values = data.loc[data.duplicated(subset=['Report'], keep=False), ['StudyInstanceUID', 'Report']]
repeated_values.reset_index(drop=True, inplace=True)
display(repeated_values)

This looks like someone copy-pasting a stock report for a diagnosis, rather than erroneously duplicated records - I'm leaving it in for now.

## 7. Sort columns

In [ ]:
# Group all *_evidence and *_agreement columns together
evidence_cols = [col for col in data.columns if col.endswith('_evidence')]
agreement_cols = [col for col in data.columns if col.endswith('_agreement')]
other_cols = [col for col in data.columns if col not in evidence_cols + agreement_cols]
data = data[other_cols + agreement_cols + evidence_cols]
data.info()

In [ ]:
data['Effusion_agreement'].value_counts()

## 6. Save cleaned data

In [ ]:
data.to_csv(config.CLEANED_CSV_PATH, index=False)